# Labor SCC Tables¶

all paper tables, pulse year 2025. produces .tex files for overleaf

versions:

uninteracted: model_collapsed (avg iiasa+oecd), iiasa only, oecd only
interacted: model_collapsed, iiasa only, oecd only (ssp3 only, both iams)
frisch mixed variants: oecd only (ssp3 only)
panel III: RFF-SP probabilistic (pulse 2025, CO2_Fossil, euler_ramsey)s

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import os

PULSE_YEAR = 2025

for root in ["/project/cil", "/Volumes/project/cil", "/Volumes/cil"]:
    if os.path.isdir(root):
        break
else:
    raise FileNotFoundError("Cannot find project mount")

PROJECT = f"{root}/home_dirs/scadavidsanchez/projects/dscim-labor-2025-update2026"
BASE = f"{PROJECT}/results"
BASE_INT = f"{PROJECT}/results_interacted_model_v2"
BASE_FR1 = f"{PROJECT}/results_frisch_0p7_0p37"
BASE_FR2 = f"{PROJECT}/results_frisch_0p37_0p7"
OUT = f"{root}/home_dirs/nishkasharma/repos/labor-code-release-2020/output/tables/"
os.makedirs(OUT, exist_ok=True)

DR = [0.015, 0.02, 0.025, 0.03, 0.05]
print(f"Pulse year: {PULSE_YEAR}")
print(f"Output: {OUT}")

# loaders and helpers

In [ ]:
def _load(path, var):
    if not os.path.exists(path):
        print(f"  NOT FOUND: {os.path.basename(path)}")
        return None
    ds = xr.open_dataset(path)
    df = ds[var].to_dataframe().reset_index()
    ds.close()
    return df

def scc_mc(base, recipe="adding_up"):
    return _load(f"{base}/scc_output/AR6_ssp/labor/{PULSE_YEAR}/unmasked/{recipe}_constant_model_collapsed_eta2.0_rho0.0_scc.nc4", "scc")

def iqr_mc(base, recipe="adding_up"):
    return _load(f"{base}/scc_output_full_uncertainty/AR6_ssp/labor/{PULSE_YEAR}/unmasked/{recipe}_constant_model_collapsed_eta2.0_rho0.0_full_uncertainty_iqr.nc4", "full_uncertainty_iqr")

def scc_bm(base, model_name, recipe="adding_up"):
    path = f"{base}/scc_output_by_model/AR6_ssp/labor/{PULSE_YEAR}/unmasked/{recipe}_constant_eta2.0_rho0.0_scc.nc4"
    if not os.path.exists(path):
        print(f"  NOT FOUND: {os.path.basename(path)}")
        return None
    ds = xr.open_dataset(path)
    df = ds["scc"].sel(model=model_name).to_dataframe().reset_index()
    ds.close()
    return df

def iqr_bm(base, model_name, recipe="adding_up"):
    path = f"{base}/scc_output_full_uncertainty_by_model/AR6_ssp/labor/{PULSE_YEAR}/unmasked/{recipe}_constant_eta2.0_rho0.0_full_uncertainty_iqr.nc4"
    if not os.path.exists(path):
        print(f"  NOT FOUND: {os.path.basename(path)}")
        return None
    ds = xr.open_dataset(path)
    df = ds["full_uncertainty_iqr"].sel(model=model_name).to_dataframe().reset_index()
    ds.close()
    return df

def decomp_load(base, dtype, vname):
    return _load(f"{base}/scc_output_decomposition/{dtype}/AR6_ssp/labor/{PULSE_YEAR}/unmasked/adding_up_constant_model_collapsed_eta2.0_rho0.0_{vname}.nc4", vname)

def P(df, ssp="SSP3", fa="median_params"):
    if df is None or df.empty:
        return pd.DataFrame(columns=["rcp","discrate","scc"])
    m = (df.ssp==ssp) & (df.weitzman_parameter.astype(str)=="0") & (df.discrate.isin(DR))
    if "fair_aggregation" in df.columns:
        m = m & (df.fair_aggregation==fa)
    return df[m][["rcp","discrate","scc"]].sort_values(["rcp","discrate"]).reset_index(drop=True)

def Q(df, ssp="SSP3", qs=(0.05,0.95)):
    if df is None or df.empty:
        lo,hi = min(qs),max(qs)
        return pd.DataFrame(columns=["rcp","discrate",f"q{lo}",f"q{hi}"])
    lo,hi = min(qs),max(qs)
    m = (df.ssp==ssp) & (df.weitzman_parameter.astype(str)=="0") & (df.discrate.isin(DR))
    sub = df[m].copy()
    sub["quantile"] = sub["quantile"].round(4)
    sub = sub[sub["quantile"].isin([round(lo,4),round(hi,4)])]
    if sub.empty:
        return pd.DataFrame(columns=["rcp","discrate",f"q{lo}",f"q{hi}"])
    vc = [c for c in sub.columns if c not in ["rcp","discrate","quantile","ssp","weitzman_parameter","discount_type","model","gas","fair_aggregation"]][0]
    w = sub.pivot_table(index=["rcp","discrate"], columns="quantile", values=vc).reset_index()
    w = w.rename(columns={round(lo,4):f"q{lo}", round(hi,4):f"q{hi}"})
    return w.sort_values(["rcp","discrate"]).reset_index(drop=True)

def V(df, rcp, dr, col="scc"):
    if df is None or df.empty: return None
    r = df[(df.rcp==rcp)&(abs(df.discrate-dr)<0.001)]
    return r.iloc[0][col] if len(r)>0 else None

def D(v):
    if v is None: return ""
    return f"\${v:.1f}"

def R(lo, hi):
    if lo is None or hi is None: return ""
    f = lambda v: f"-\${abs(v):.1f}" if v<0 else f"\${v:.1f}"
    return f"[{f(lo)},{f(hi)}]"

def wtex(name, lines):
    path = f"{OUT}/{name}.tex"
    with open(path, "w") as f:
        f.write("\n".join(lines))
    print(f"  saved: {name}.tex")

# load all data

In [ ]:
print("uninteracted: model_collapsed")
au_mc = scc_mc(BASE, "adding_up")
ra_mc = scc_mc(BASE, "risk_aversion")
iq_mc = iqr_mc(BASE)
dfunc_u = decomp_load(BASE, "damage_fn_uncertainty", "stat_uncertainty_iqr")
clim_u = decomp_load(BASE, "climate_uncertainty", "climate_uncertainty_iqr")

print("\ninteracted: model_collapsed")
au_int_mc = scc_mc(BASE_INT, "adding_up")
ra_int_mc = scc_mc(BASE_INT, "risk_aversion")
iq_int_mc = iqr_mc(BASE_INT)
dfunc_i = decomp_load(BASE_INT, "damage_fn_uncertainty", "stat_uncertainty_iqr")
clim_i = decomp_load(BASE_INT, "climate_uncertainty", "climate_uncertainty_iqr")

print("\ndone.")

# table 5: SSP 3 SCC

In [ ]:
def make_table5(au, ra, iq, label, tag, ssp="SSP3"):
    p = P(au, ssp); r = Q(iq, ssp, (0.01,0.99)); ce = P(ra, ssp, "ce")
    if p.empty:
        print(f"table 5 {label}: no data, skipping")
        return
    rows = []
    for rcp in ["rcp85","rcp45"]:
        rl = "RCP 8.5" if rcp=="rcp85" else "RCP 4.5"
        for dr in DR:
            sv=V(p,rcp,dr); cv=V(ce,rcp,dr)
            rr=r[(r.rcp==rcp)&(abs(r.discrate-dr)<0.001)]
            lo=rr.iloc[0]["q0.01"] if len(rr)>0 else None
            hi=rr.iloc[0]["q0.99"] if len(rr)>0 else None
            rows.append({"RCP":rl,"dr":f"{dr*100:.1f}%","SCC":f"${sv:.1f}" if sv else "","1-99%":f"[{lo:.1f},{hi:.1f}]" if lo else "","CE":f"${cv:.1f}" if cv else ""})
    print(f"table 5: {label}")
    display(pd.DataFrame(rows))
    L = []
    L.append(r"\begin{table}[!htbp]")
    L.append(r"\centering")
    L.append(r"{\renewcommand{\arraystretch}{1.05}")
    L.append(r"\scriptsize{\begin{tabular}{>{\centering\arraybackslash}lccccc}")
    L.append(r"&&\\")
    L.append(r"\textbf{Discount rate} & $\delta = 1.5\%$ & $\delta = 2\%$ & $\delta = 2.5\%$ & $\delta = 3\%$ & $\delta = 5\%$\\")
    L.append(r"\hline")
    L.append(r"&&&&&\\")
    L.append(r"&\multicolumn{5}{c}{\textbf{I. Partial SCC estimates}}\\")
    L.append(r"&&&&&\\")
    L.append(r"&\multicolumn{5}{c}{\textbf{" + ssp + " " + label + r"}}\\")
    for rcp, rl in [("rcp85","RCP 8.5"),("rcp45","RCP 4.5")]:
        sv=" & ".join([D(V(p,rcp,dr)) for dr in DR])
        rv=" & ".join([R(V(r,rcp,dr,"q0.01"),V(r,rcp,dr,"q0.99")) for dr in DR])
        L.append(r"\hspace{0mm}\multirow{2}{*}{\textbf{" + rl + r"}} & " + sv + r" \\")
        L.append(r"\hspace{0mm} & " + rv + r" \\")
    L.append(r"&&&&&\\")
    L.append(r"&\multicolumn{5}{c}{\textbf{II. Certainty equivalent}}\\")
    L.append(r"&&&&&\\")
    for rcp, rl in [("rcp85","RCP 8.5"),("rcp45","RCP 4.5")]:
        cv=" & ".join([D(V(ce,rcp,dr)) for dr in DR])
        L.append(r"\hspace{0mm}\textbf{" + rl + r"} & " + cv + r" \\")
    L.append(r"\hline")
    L.append(r"\end{tabular}}}")
    L.append(r"\caption{\textbf{Partial SCC for labor disutility} " + label + r". Pulse year " + str(PULSE_YEAR) + r".}")
    L.append(r"\label{tab:t5_" + tag + r"}")
    L.append(r"\end{table}")
    wtex(f"scc_table5_{tag}", L)

make_table5(au_mc, ra_mc, iq_mc, "uninteracted, model collapsed", "unint_mc")
make_table5(au_int_mc, ra_int_mc, iq_int_mc, "interacted, model collapsed", "int_mc")

# panel III: RFF probabilistic approach

pulse year 2025, CO2_Fossil only, euler_ramsey discounting with 10k RFF-SP draws. uses adding_up mean (risk_aversion gives nearly identical values at these low etas).

In [ ]:
# panel III: RFF probabilistic (CO2_Fossil, weitzman=0)
# uses adding_up mean (risk_aversion gives nearly identical values at these low etas)

print(f"Panel III: RFF-SP probabilistic approach (pulse {PULSE_YEAR})")
eta_rho = [
    (1.016010255, 9.149608e-05, "1.5%"),
    (1.244459066, 0.00197263997, "2.0%"),
    (1.421158116, 0.00461878399, "2.5%"),
]
rff_vals = {}
for eta, rho, label in eta_rho:
    path = f"{BASE}/scc_output/rff_results/labor/{PULSE_YEAR}/unmasked/adding_up_euler_ramsey_eta{eta}_rho{rho}_uncollapsed_sccs.nc4"
    if not os.path.exists(path):
        print(f"  {label}: NOT FOUND")
        continue
    ds = xr.open_dataset(path)
    scc = ds["uncollapsed_sccs"].sel(gas="CO2_Fossil", weitzman_parameter="0")
    rff_vals[label] = float(scc.mean())
    print(f"  {label}: ${rff_vals[label]:.1f}")
    ds.close()

if not rff_vals:
    print("  no RFF results found, skipping Panel III")

# table h.1: uncertainty decomposition

In [ ]:
def make_h1(au, iq, df_iq, cl_iq, label, tag):
    ph=P(au); rfull=Q(iq,qs=(0.01,0.99))
    if ph.empty:
        print(f"table h.1 {label}: no data, skipping")
        return
    rdf=Q(df_iq,qs=(0.01,0.99)) if df_iq is not None else pd.DataFrame()
    rcl=Q(cl_iq,qs=(0.01,0.99)) if cl_iq is not None else pd.DataFrame()
    rows = []
    for rcp in ["rcp85","rcp45"]:
        rl = "RCP 8.5" if rcp=="rcp85" else "RCP 4.5"
        for dr in DR:
            sv=V(ph,rcp,dr)
            fl=rfull[(rfull.rcp==rcp)&(abs(rfull.discrate-dr)<0.001)]
            dl=rdf[(rdf.rcp==rcp)&(abs(rdf.discrate-dr)<0.001)] if not rdf.empty else pd.DataFrame()
            cl=rcl[(rcl.rcp==rcp)&(abs(rcl.discrate-dr)<0.001)] if not rcl.empty else pd.DataFrame()
            rows.append({"RCP":rl,"dr":f"{dr*100:.1f}%","Point":f"${sv:.1f}" if sv else "",
                "Full":f"[{fl.iloc[0]['q0.01']:.1f},{fl.iloc[0]['q0.99']:.1f}]" if len(fl)>0 else "",
                "DmgFn":f"[{dl.iloc[0]['q0.01']:.1f},{dl.iloc[0]['q0.99']:.1f}]" if len(dl)>0 else "",
                "Climate":f"[{cl.iloc[0]['q0.01']:.1f},{cl.iloc[0]['q0.99']:.1f}]" if len(cl)>0 else ""})
    print(f"table h.1: {label}")
    display(pd.DataFrame(rows))
    L = []
    L.append(r"\begin{table}[!htbp]")
    L.append(r"\centering")
    L.append(r"{\renewcommand{\arraystretch}{1.05}")
    L.append(r"\scriptsize{\begin{tabular}{>{\centering\arraybackslash}lccccc}")
    L.append(r"&&\\")
    L.append(r"\textbf{Discount rate} & $\delta = 1.5\%$ & $\delta = 2\%$ & $\delta = 2.5\%$ & $\delta = 3\%$ & $\delta = 5\%$\\")
    L.append(r"\hline")
    for rcp, rl in [("rcp85","RCP 8.5"),("rcp45","RCP 4.5")]:
        L.append(r"&&&&&\\")
        pl=" & ".join([D(V(ph,rcp,dr)) for dr in DR])
        L.append(r"\textbf{" + rl + r"} & " + pl + r" \\")
        fl=" & ".join([R(V(rfull,rcp,dr,"q0.01"),V(rfull,rcp,dr,"q0.99")) for dr in DR])
        L.append(r"\quad Full uncertainty & " + fl + r" \\")
        if not rdf.empty:
            dl=" & ".join([R(V(rdf,rcp,dr,"q0.01"),V(rdf,rcp,dr,"q0.99")) for dr in DR])
            L.append(r"\quad Damage fn only & " + dl + r" \\")
        if not rcl.empty:
            cl=" & ".join([R(V(rcl,rcp,dr,"q0.01"),V(rcl,rcp,dr,"q0.99")) for dr in DR])
            L.append(r"\quad Climate sens only & " + cl + r" \\")
    L.append(r"\hline")
    L.append(r"\end{tabular}}}")
    L.append(r"\caption{\textbf{Uncertainty decomposition} " + label + r". Pulse year " + str(PULSE_YEAR) + r".}")
    L.append(r"\label{tab:h1_" + tag + r"}")
    L.append(r"\end{table}")
    wtex(f"scc_table_h1_{tag}", L)

make_h1(au_mc, iq_mc, dfunc_u, clim_u, "uninteracted, model collapsed", "uninteracted")

# table h.2: alternative ssps

In [ ]:
def make_h2(au, ra, iq, label, tag):
    rows = []
    for ssp in ["SSP2","SSP3","SSP4"]:
        ps=P(au,ssp=ssp)
        if ps.empty:
            print(f"  {ssp}: no data, skipping")
            continue
        rs=Q(iq,ssp=ssp,qs=(0.01,0.99)); cs=P(ra,ssp=ssp,fa="ce")
        for rcp in ["rcp85","rcp45"]:
            rl = "RCP 8.5" if rcp=="rcp85" else "RCP 4.5"
            for dr in DR:
                sv=V(ps,rcp,dr); cv=V(cs,rcp,dr)
                rr=rs[(rs.rcp==rcp)&(abs(rs.discrate-dr)<0.001)]
                rows.append({"SSP":ssp,"RCP":rl,"dr":f"{dr*100:.1f}%",
                    "SCC":f"${sv:.1f}" if sv else "","1-99%":f"[{rr.iloc[0]['q0.01']:.1f},{rr.iloc[0]['q0.99']:.1f}]" if len(rr)>0 else "",
                    "CE":f"${cv:.1f}" if cv else ""})
    if not rows:
        print(f"table h.2 {label}: no data for any ssp, skipping")
        return
    print(f"table h.2: {label}")
    display(pd.DataFrame(rows))
    L = []
    L.append(r"\begin{table}[!htbp]")
    L.append(r"\centering")
    L.append(r"{\renewcommand{\arraystretch}{1.05}")
    L.append(r"\scriptsize{\begin{tabular}{>{\centering\arraybackslash}lccccc}")
    L.append(r"&&\\")
    L.append(r"\textbf{Discount rate} & $\delta = 1.5\%$ & $\delta = 2\%$ & $\delta = 2.5\%$ & $\delta = 3\%$ & $\delta = 5\%$\\")
    L.append(r"\hline")
    L.append(r"&&&&&\\")
    L.append(r"&\multicolumn{5}{c}{\textbf{I. Partial SCC estimates}}\\")
    for ssp in ["SSP2","SSP3","SSP4"]:
        ps=P(au,ssp=ssp)
        if ps.empty: continue
        L.append(r"&&&&&\\")
        L.append(r"&\multicolumn{5}{c}{\textbf{" + ssp + r"}}\\")
        rs=Q(iq,ssp=ssp,qs=(0.01,0.99))
        for rcp, rl in [("rcp85","RCP 8.5"),("rcp45","RCP 4.5")]:
            sv=" & ".join([D(V(ps,rcp,dr)) for dr in DR])
            rv=" & ".join([R(V(rs,rcp,dr,"q0.01"),V(rs,rcp,dr,"q0.99")) for dr in DR])
            L.append(r"\hspace{0mm}\multirow{2}{*}{\textbf{" + rl + r"}} & " + sv + r" \\")
            L.append(r"\hspace{0mm} & " + rv + r" \\")
    L.append(r"&&&&&\\")
    L.append(r"&\multicolumn{5}{c}{\textbf{II. Certainty equivalent}}\\")
    for ssp in ["SSP2","SSP3","SSP4"]:
        cs=P(ra,ssp=ssp,fa="ce")
        if cs.empty: continue
        L.append(r"&&&&&\\")
        L.append(r"&\multicolumn{5}{c}{\textbf{" + ssp + r"}}\\")
        for rcp, rl in [("rcp85","RCP 8.5"),("rcp45","RCP 4.5")]:
            cv=" & ".join([D(V(cs,rcp,dr)) for dr in DR])
            L.append(r"\hspace{0mm}\textbf{" + rl + r"} & " + cv + r" \\")
    L.append(r"\hline")
    L.append(r"\end{tabular}}}")
    L.append(r"\caption{\textbf{Alternative socioeconomic scenarios} " + label + r". Pulse year " + str(PULSE_YEAR) + r".}")
    L.append(r"\label{tab:h2_" + tag + r"}")
    L.append(r"\end{table}")
    wtex(f"scc_table_h2_{tag}", L)

make_h2(au_mc, ra_mc, iq_mc, "uninteracted, model collapsed", "uninteracted")

# summary

In [ ]:
print(f"output: {OUT}")
for f in sorted(os.listdir(OUT)):
    if f.endswith(".tex"):
        print(f"  {f}")